18.4 secs

## Load libraries

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
from collections import defaultdict

## Config

In [10]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# GB tuning grid
arx_param_grid = {
    "alpha": [0.0, 0.1, 1.0, 10.0, 100.0],  # 0.0 = OLS
}

## Metrics

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()
df_tv["LA_ID"] = df_tv[ENTITY_COL].astype("category")

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Rolling STL feature builder

In [13]:
def add_rolling_stl_components(
    df: pd.DataFrame,
    entity_col: str,
    time_col: str,
    target_col: str,
    period: int = 12,
    min_history: int = 24,
    robust: bool = True,
    show_progress: bool = True,
) -> pd.DataFrame:
    """
    Time-safe rolling STL (one-sided).
    For each entity and each time t, fit STL on y[:t] and assign the last component values to time t.

    Outputs columns:
      - stl_trend
      - stl_seasonal
      - stl_resid

    Notes:
    - This is computationally heavier than "fit once on train then extrapolate".
    - It avoids leakage because STL at time t uses only <= t data.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    grouped = df.groupby(entity_col, sort=False)
    iterator = grouped if not show_progress else tqdm(grouped, desc="Rolling STL by LA", leave=False)

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        idx = sub.index.values

        # Rolling one-sided STL: start only when we have enough history
        for t in range(min_history - 1, len(y)):
            y_hist = y[: t + 1]

            # Skip if history contains NaNs
            if np.isnan(y_hist).any():
                continue

            try:
                stl = STL(y_hist, period=period, robust=robust)
                res = stl.fit()

                df.loc[idx[t], "stl_trend"] = float(res.trend[-1])
                df.loc[idx[t], "stl_seasonal"] = float(res.seasonal[-1])
                df.loc[idx[t], "stl_resid"] = float(res.resid[-1])

            except Exception:
                # If STL fails for numeric reasons at this t, leave NaNs
                continue

    return df

## Training with STL and rolling CV

In [16]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    fold_specs.append((
        dates[train_end_idx - ROLLING_TRAIN_WINDOW],
        dates[train_end_idx - 1],
        dates[val_start_idx],
        dates[val_end_idx - 1],
    ))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# -----------------------------
# CV loop
# -----------------------------

for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, 1):
    print(f"\n=== Fold {fold_no}: Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")

    fold_train = df_tv[(df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)].copy()
    fold_val   = df_tv[(df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)].copy()

    fold_train["is_train"] = True
    fold_val["is_train"] = False

    combined = pd.concat([fold_train, fold_val], axis=0).sort_values([ENTITY_COL, TIME_COL])

    # ---- TRUE rolling STL as-of-time (no extrapolation) ----
    combined = add_rolling_stl_components(
        combined,
        entity_col=ENTITY_COL,
        time_col=TIME_COL,
        target_col=TARGET_COL,
        period=12,
        min_history=24,
        robust=True,
        show_progress=True,   # set True if you want tqdm every fold
    )

    # ---- create STL lags for ALL lags you might use ----
    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    # -----------------------------
    # Lag-set loop
    # -----------------------------
    for lag_set in lag_combinations:
        lag_cols = [f"{c}_lag{l}" for c in ["stl_trend", "stl_seasonal", "stl_resid"] for l in lag_set]
        feature_cols = continuous_cols + categorical_cols + lag_cols

        train_df = combined[combined["is_train"]].copy()
        val_df   = combined[~combined["is_train"]].copy()

        # Drop rows missing features/target (rolling STL introduces NaNs early)
        train_df = train_df.dropna(subset=feature_cols + [TARGET_COL])
        val_df   = val_df.dropna(subset=feature_cols + [TARGET_COL])

        if train_df.empty or val_df.empty:
            continue

        # Scale only continuous + STL-lag features (not one-hot categoricals)
        scale_cols = continuous_cols + lag_cols
        scaler = StandardScaler()
        train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
        val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))

        X_train = train_df[feature_cols]
        y_train = train_df[TARGET_COL].values.astype(float)

        X_val   = val_df[feature_cols]
        y_val   = val_df[TARGET_COL].values.astype(float)

        for params in ParameterGrid(arx_param_grid):
            key = (tuple(lag_set), tuple(sorted(params.items())))
            metrics_store.setdefault(key, {
                "lag_set": tuple(lag_set),
                "params": params,
                "mae": [], "rmse": [], "smape": [], "mase": [], "folds": 0
            })

            model = Ridge(alpha=float(params["alpha"]), fit_intercept=True)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            m = metrics_store[key]
            m["mae"].append(mae(y_val, y_pred))
            m["rmse"].append(rmse(y_val, y_pred))
            m["smape"].append(smape(y_val, y_pred))
            m["mase"].append(mase(y_val, y_pred, y_train, m=12))
            m["folds"] += 1

Number of folds: 5

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===


C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is d


=== Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03 ===


C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.20920938 -1.20920938 -1.20920938 ...  4.58937109  4.58937109
  4.58937109]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.20920938 -1.20920938 -1.20920938 ...  4.58937109  4.58937109
  4.58937109]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is d


=== Fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03 ===


C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.31191135 -1.31191135 -1.31191135 ...  4.18066929  4.18066929
  4.18066929]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.31191135 -1.31191135 -1.31191135 ...  4.18066929  4.18066929
  4.18066929]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is d


=== Fold 4: Train 2010-04–2020-03, Val 2020-04–2021-03 ===


C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.44398077 -1.44398077 -1.44398077 ...  3.92382577  3.92382577
  3.92382577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.44398077 -1.44398077 -1.44398077 ...  3.92382577  3.92382577
  3.92382577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is d


=== Fold 5: Train 2011-04–2021-03, Val 2021-04–2022-03 ===


C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.34307846 -1.34307846 -1.34307846 ...  3.68968969  3.68968969
  3.68968969]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, scale_cols] = scaler.fit_transform(train_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.34307846 -1.34307846 -1.34307846 ...  3.68968969  3.68968969
  3.68968969]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, scale_cols]   = scaler.transform(val_df[scale_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_10708\690602840.py:84: FutureWarning: Setting an item of incompatible dtype is d

## Results

In [18]:
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "ARX_Ridge",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/arx_leakfree_rollingcv_results.csv", index=False)

   model_type                     lag_set           params  folds  \
0   ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)   {'alpha': 0.0}      5   
1   ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)   {'alpha': 0.1}      5   
2   ARX_Ridge  (1, 2, 3, 4, 5, 6, 12, 24)   {'alpha': 0.0}      5   
3   ARX_Ridge  (1, 2, 3, 4, 5, 6, 12, 24)   {'alpha': 0.1}      5   
4   ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)   {'alpha': 1.0}      5   
5   ARX_Ridge  (1, 2, 3, 4, 5, 6, 12, 24)   {'alpha': 1.0}      5   
6   ARX_Ridge               (1, 2, 3, 12)   {'alpha': 0.0}      5   
7   ARX_Ridge                  (1, 2, 12)   {'alpha': 0.1}      5   
8   ARX_Ridge                  (1, 2, 12)   {'alpha': 0.0}      5   
9   ARX_Ridge               (1, 2, 3, 12)   {'alpha': 0.1}      5   
10  ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)  {'alpha': 10.0}      5   
11  ARX_Ridge                  (1, 2, 12)   {'alpha': 1.0}      5   
12  ARX_Ridge               (1, 2, 3, 12)   {'alpha': 1.0}      5   
13  ARX_Ridge           (1, 2, 3, 

In [19]:
df_tv[TARGET_COL].describe()

count    5.292000e+04
mean     2.361938e+05
std      1.372830e+05
min      6.367500e+04
25%      1.462590e+05
50%      1.989920e+05
75%      2.820110e+05
max      1.653252e+06
Name: AveragePrice, dtype: float64